# Catalog ingestion workflow

This notebook shows how to **ingest a survey catalog** (FITS or VOTable with Astropy) into HEALPix-partitioned Parquet under `catalogs/<survey>/`, validate the result, and query it with `CatalogAccessor`.

**Two modes**

1. **Self-contained demo** — writes a tiny synthetic FITS into a temporary lake (no `/data/lake` needed).
2. **Your data** — set `USE_TEMP_LAKE = False` and point `LAKE_ROOT` / `CATALOG_FITS` at your deployment and input file.

For huge catalogs (tens of millions of rows), use the **`--streaming`** flag on `dl-ingest-catalog` or `ingest_catalog(..., streaming=True)` so memory stays bounded (see README).

In [ ]:
from __future__ import annotations

import json
import shutil
import tempfile
from pathlib import Path

import numpy as np
from astropy.table import Table

from data_lake.ingest.fits_to_parquet import ingest_catalog
from data_lake.ingest.validate_catalog_ingest import run_validation
from data_lake.io.catalog import CatalogAccessor

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

USE_TEMP_LAKE = True
SURVEY = "demo_catalog_nb"
NORDER = 5

if USE_TEMP_LAKE:
    LAKE_ROOT = Path(tempfile.mkdtemp(prefix="data_lake_catalog_nb_"))
    CATALOG_FITS = LAKE_ROOT / "input" / "mini_catalog.fits"
else:
    LAKE_ROOT = Path("/data/lake")  # your deployment data root (see lake_config.toml [lake].root)
    CATALOG_FITS = Path("/data/raw/my_survey.fits")  # your FITS / VOTable path

RA_COL = "ra"
DEC_COL = "dec"
SOURCE_ID_COL = "source_id"  # FITS column with int64 IDs; use e.g. TARGETID for DESI

print("LAKE_ROOT:", LAKE_ROOT.resolve())
print("CATALOG_FITS:", CATALOG_FITS)

## 1. (Demo only) Write a small FITS catalog

Skip this cell when `USE_TEMP_LAKE = False` and your `CATALOG_FITS` already exists.

The ingestor expects sky columns (`ra_col`, `dec_col`) and either a **`source_id`** column or you pass `source_id_col=` to map an existing integer ID column.

In [ ]:
if USE_TEMP_LAKE:
    rng = np.random.default_rng(0)
    n = 800
    tbl = Table(
        {
            "source_id": np.arange(1, n + 1, dtype=np.int64),
            "ra": rng.uniform(200.0, 201.0, n),
            "dec": rng.uniform(-0.5, 0.5, n),
            "mag_g": rng.uniform(18.0, 24.0, n).astype(np.float32),
        }
    )
    CATALOG_FITS.parent.mkdir(parents=True, exist_ok=True)
    tbl.write(CATALOG_FITS, format="fits", overwrite=True)
    print("Wrote", CATALOG_FITS, "rows=", len(tbl))
else:
    assert CATALOG_FITS.is_file(), f"Missing catalog file: {CATALOG_FITS}"

## 2. Ingest into the lake (`ingest_catalog`)

This writes `catalogs/<survey>/Norder=…/Dir=…/Npix=….parquet`, aggregate `_metadata`, and `catalog_info.json`.

- **`overwrite=False`** (default): existing tile files are left unchanged — useful for incremental fills; set `overwrite=True` to rebuild tiles that already exist.
- **`streaming=True`**: use for very large FITS so only one HEALPix tile is materialised at a time (slower wall-clock, much lower RAM).

In [ ]:
ingest_catalog(
    source_path=CATALOG_FITS,
    output_root=LAKE_ROOT,
    survey_name=SURVEY,
    ra_col=RA_COL,
    dec_col=DEC_COL,
    norder=NORDER,
    source_id_col=SOURCE_ID_COL,
    overwrite=True,
    streaming=False,
)

cat_root = LAKE_ROOT / "catalogs" / SURVEY
info = json.loads((cat_root / "catalog_info.json").read_text())
print("catalog_info keys:", sorted(info.keys()))
print("total_rows (metadata):", info.get("total_rows"))

## 3. Validate the survey directory

`run_validation` checks `catalog_info.json`, optional ingest sidecars, aggregate `_metadata`, and a sample of Parquet tiles for required columns (`source_id`, RA/Dec, HEALPix column).

In [ ]:
rep = run_validation(LAKE_ROOT, SURVEY, max_tiles=50)
print("errors:", rep.errors)
print("warnings:", rep.warnings)
assert rep.ok(strict=False), "Validation failed — fix errors above before production ingest."

## 4. Query with `CatalogAccessor`

DuckDB reads only the Parquet columns touched by your SQL (projection pushdown).

In [ ]:
with CatalogAccessor(LAKE_ROOT, SURVEY, norder=NORDER) as cat:
    print(cat)
    df = cat.query(
        f"""
        SELECT source_id, {RA_COL}, {DEC_COL}, mag_g
        FROM catalog
        WHERE mag_g < 20.5
        ORDER BY mag_g
        LIMIT 10
        """
    )
df

## 5. CLI equivalents (terminal)

With `DATA_LAKE_CONFIG` pointing at your deployment, `OUTPUT_ROOT` is optional:

```bash
export DATA_LAKE_CONFIG=/path/to/my_lake/lake_config.toml

dl-ingest-catalog /data/raw/my_survey.fits --survey my_survey \\
  --ra-col RA --dec-col DEC --source-id-col TARGETID

# Huge FITS (e.g. DESI zall-pix): add --streaming
dl-ingest-catalog zall-pix-iron.fits --survey desi_dr1 \\
  --ra-col TARGET_RA --dec-col TARGET_DEC --source-id-col TARGETID --streaming

dl-validate-catalog-ingest --survey my_survey
```

Batch many FITS files with checkpoints:

```bash
find /data/cats -name '*.fits' > cat_files.txt
dl-ingest-catalog-from-list cat_files.txt --survey my_survey --ra-col RA --dec-col DEC
```

## 6. (Optional) Resolve `LAKE_ROOT` from `lake_config.toml`

If you prefer not to hardcode paths, use the same discovery order as the `dl-*` CLIs (`$DATA_LAKE_CONFIG`, then walk upward from the current working directory):

```python
from data_lake.config import LakeConfig
cfg = LakeConfig.discover()
LAKE_ROOT = cfg.lake.root
```

## 7. (Demo only) Remove temporary lake

Skip if `USE_TEMP_LAKE = False`.

In [ ]:
if USE_TEMP_LAKE:
    shutil.rmtree(LAKE_ROOT, ignore_errors=True)
    print("Removed temp lake:", LAKE_ROOT)